# Extract oMST REC and LDI scores using subject ID

This notebook scans a JATOS results zip or extracted folder, finds each `data.txt`,
uses the `subject` field inside the TXT as the participant ID,
extracts the final `REC` and `LDI` values from the oMST summary,
and saves a combined CSV.


In [1]:
from pathlib import Path
import zipfile
import re
import pandas as pd

INPUT_PATH = Path('/Users/jianleguo/Desktop/Mario_Pilot_Data/clean_use/oMST/jatos_results_data_20260328022308.zip')
OUTPUT_DIR = Path('/Users/jianleguo/Desktop/Mario_Pilot_Data/clean_use/oMST/data')
OUTPUT_CSV = OUTPUT_DIR / 'combined_oMST_result_by_subject.csv'

SUBJECT_PATTERN = re.compile(r'"subject"\s*:\s*"([^"]+)"')
REC_LDI_PATTERN = re.compile(r'REC,\s*([0-9]*\.?[0-9]+)\s*,\s*LDI,\s*([0-9]*\.?[0-9]+)')


In [2]:
def list_data_files(input_path: Path):
    if input_path.is_file() and input_path.suffix.lower() == '.zip':
        with zipfile.ZipFile(input_path, 'r') as zf:
            return [name for name in zf.namelist() if name.endswith('data.txt')]
    elif input_path.is_dir():
        return sorted(input_path.rglob('data.txt'))
    else:
        raise FileNotFoundError(f'Cannot find input: {input_path}')

def read_text(input_path: Path, file_ref):
    if input_path.is_file() and input_path.suffix.lower() == '.zip':
        with zipfile.ZipFile(input_path, 'r') as zf:
            return zf.read(file_ref).decode('utf-8', errors='replace')
    return Path(file_ref).read_text(encoding='utf-8', errors='replace')

def extract_subject(text: str):
    matches = SUBJECT_PATTERN.findall(text)
    return matches[-1] if matches else None

def extract_rec_ldi(text: str):
    matches = REC_LDI_PATTERN.findall(text)
    if not matches:
        return None
    rec, ldi = matches[-1]
    return float(rec), float(ldi)


In [3]:
rows = []
skipped = []

for file_ref in list_data_files(INPUT_PATH):
    text = read_text(INPUT_PATH, file_ref)
    subject = extract_subject(text)
    score = extract_rec_ldi(text)

    if subject is None or score is None:
        skipped.append(str(file_ref))
        continue

    rec, ldi = score
    rows.append({
        'participant_id': subject,
        'REC': rec,
        'LDI': ldi,
    })

df = pd.DataFrame(rows).drop_duplicates(subset=['participant_id']).sort_values('participant_id').reset_index(drop=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
df.to_csv(OUTPUT_CSV, index=False)

print(f'Saved {len(df)} rows to: {OUTPUT_CSV}')
print(f'Skipped files: {len(skipped)}')
df.head(10)


Saved 47 rows to: /Users/jianleguo/Desktop/Mario_Pilot_Data/clean_use/oMST/data/combined_oMST_result_by_subject.csv
Skipped files: 193


,participant_id,REC,LDI
0,55bb9ae7fdf99b26d27fda01,0.650,0.469
1,56514c84715ff50011a43878,0.750,0.636
2,5cb9ca6814b3cb0017e4f7dd,0.650,0.825
3,5d4c78071b920f00198e8533,0.400,0.575
4,5e9cbc3999531a099492509f,0.784,0.452
5,5ea37c11733b42302b54b64c,0.850,0.901
6,5f0d2ad60ad0d6730c380b05,0.650,0.729
7,5fa4c21a04bc5912180291bc,0.650,0.655
8,603006ea5f6c24dfc908f835,0.900,0.523
9,60e144a275d075549f49d12f,0.834,0.438


In [4]:
df


,participant_id,REC,LDI
0,55bb9ae7fdf99b26d27fda01,0.650,0.469
1,56514c84715ff50011a43878,0.750,0.636
2,5cb9ca6814b3cb0017e4f7dd,0.650,0.825
3,5d4c78071b920f00198e8533,0.400,0.575
4,5e9cbc3999531a099492509f,0.784,0.452
5,5ea37c11733b42302b54b64c,0.850,0.901
6,5f0d2ad60ad0d6730c380b05,0.650,0.729
7,5fa4c21a04bc5912180291bc,0.650,0.655
8,603006ea5f6c24dfc908f835,0.900,0.523
9,60e144a275d075549f49d12f,0.834,0.438
